# XAI-RiceGuard — Phase 02: Baseline Architectures & Model Training Pipeline

**Project:** A Lesion-Grounded Explainable and Uncertainty-Aware Deep Learning Framework for Robust Rice Leaf Blast and Brown Spot Detection  
**Target Publication:** IEEE-oriented Research Journal / Conference  
**Objective:** Establish reproducible, standardized training and validation for 3 core baseline architectures:
1. **EfficientNet-B0** (Compound-scaled inverted residual CNN)
2. **ResNet-50** (Conventional residual network)
3. **ConvNeXt-Tiny** (Modernized pure ConvNet)

### Strict Data Firewall Rules:
- **Training:** Uses ONLY `manifests/phase1/primary_train.csv` (12,568 images)
- **Validation / Model Selection:** Uses ONLY `manifests/phase1/primary_validation.csv` (1,798 images)
- **Calibration (`primary_calibration.csv`):** **LOCKED & UNTOUCHED** (Phase 08)
- **Internal Test (`primary_internal_test.csv`):** **LOCKED & UNTOUCHED** (Final Evaluation)
- **External Benchmarks (`sethy_external.csv`, `bd5_external.csv`, `riceseg_ground_truth.csv`):** **STRICTLY FORBIDDEN**

## 1. Mount Google Drive & Environment Detection

In [ ]:
import os
import sys

# Check if running on Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/BTech Final Year Project/XAI-RiceGuard'
    if not os.path.exists(project_path):
        project_path = '/content/XAI-Rice'
    os.chdir(project_path)
    print(f"Working in: {os.getcwd()}")
except ImportError:
    print("Running locally / non-Colab environment.")

sys.path.insert(0, os.getcwd())

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Verify Phase 01 Frozen Manifests & Data Firewall

In [ ]:
import json
from src.phase2.dataset import verify_manifest_firewall, CANONICAL_CLASSES

with open("manifests/phase1/manifest_checksums.json", "r") as f:
    checksums = json.load(f)

print("Phase 01 Manifest Checksums:")
for k, v in checksums.items():
    print(f"  {k:<30}: {v[:16]}...")

# Verify Firewall
verify_manifest_firewall("manifests/phase1/primary_train.csv", expected_role="train")
verify_manifest_firewall("manifests/phase1/primary_validation.csv", expected_role="validation")
print("\nData Firewall Verified: PASS")
print(f"Canonical 6 Classes: {CANONICAL_CLASSES}")

## 3. Rapid Smoke Test (1 Mini-Epoch)

In [ ]:
!python scripts/run_phase2_training.py --smoke-test

## 4. Train All 3 Baseline Architectures (Sequential with Auto-Resume)
Runs **EfficientNet-B0**, **ResNet-50**, and **ConvNeXt-Tiny** with 30 epochs, AdamW, and Cosine Annealing scheduler.
If interrupted, re-running this cell automatically resumes from `last_checkpoint.pt` without losing progress.

In [ ]:
!python scripts/run_phase2_training.py --all-baselines --epochs 30 --batch-size 32

## 5. Model Comparison, Validation Results & Winner Selection

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

comp_df = pd.read_csv("results/phase2/baseline_comparison.csv")
display(comp_df)

with open("reports/phase2/phase2_model_comparison.md", "r") as f:
    report_md = f.read()
display(Markdown(report_md))

## 6. Display Comparison Figures

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig_paths = [
    "figures/phase2/model_comparison.png",
    "experiments/phase2/efficientnet_b0/figures/confusion_matrix.png",
    "experiments/phase2/resnet50/figures/confusion_matrix.png",
    "experiments/phase2/convnext_tiny/figures/confusion_matrix.png"
]

for p in fig_paths:
    if os.path.exists(p):
        img = Image.open(p)
        plt.figure(figsize=(9, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(os.path.basename(p))
        plt.show()